# Entrenar el modelo de texto (cuentos) en local — RTX 4090

Afina un **T5 en español ya preentrenado** con nuestro dataset de cuentos, condicionado por temática.
Es seq2seq: la entrada del encoder es `"escribe un cuento sobre <tematica>"` y la salida del decoder es el cuento.

**Notas para la RTX 4090 (24 GB, Ada Lovelace):**
- La 4090 **sí soporta bf16**, que es estable para T5/mT5 (a diferencia de fp16, que da `NaN`). Entrenamos en **bf16 mixto** → más rápido y menos memoria.
- Modelo por defecto: `google/mt5-base` (~580M). Con 24 GB entra cómodo con `LOTE=16`.
- Alternativas: `google/mt5-small` (~300M, más rápido) o `google/mt5-large` (~1.2B, máxima calidad — activa `GRADIENT_CHECKPOINTING` y baja `LOTE`).
- El código **auto-detecta bf16**: si lo corres en una GPU sin soporte (p.ej. una 2070) cae a fp32 solo.

Este notebook es autocontenido: no importa los scripts del repo, trae las funciones dentro.

## Paso 0 — Verificar que la GPU está disponible

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
    print("Soporta bf16:", torch.cuda.is_bf16_supported())
else:
    print("AVISO: no se detecta CUDA. Revisa el driver de NVIDIA en Windows y la instalacion de torch con CUDA.")

## Paso 1 — Instalar dependencias (corre una sola vez por entorno)

Si ya creaste el venv e instalaste con `pip install -r requirements.txt` desde la terminal, puedes saltarte esta celda. `sentencepiece` es **obligatorio** para el tokenizer de T5.

In [ ]:
# Descomenta si te falta algo. torch con CUDA mejor instalarlo desde la terminal (ver pasos del README de abajo).
# %pip install transformers sentencepiece pandas tqdm

## Paso 2 — Configuración

Ajusta `RUTA_REPO` a donde clonaste el repo. En WSL, si el repo está en Windows sería algo como `/mnt/c/Users/<tu_usuario>/...`; si lo copiaste al home de Linux (recomendado por velocidad), algo como `/home/<tu_usuario>/cuentos-ilustrados-ia`.

In [ ]:
import os

# Por defecto asume que el notebook vive en <repo>/notebooks/
RUTA_REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))
# Si no, descomenta y pon la ruta a mano:
# RUTA_REPO = "/home/dyolt/cuentos-ilustrados-ia"

RUTA_PARCIALES = os.path.join(RUTA_REPO, "datos", "cuentos", "parciales")
RUTA_DATASET   = os.path.join(RUTA_REPO, "datos", "cuentos", "dataset_cuentos.csv")
RUTA_SALIDA    = os.path.join(RUTA_REPO, "modelos", "texto")

# --- Modelo e hiperparametros (configurado para RTX 4090, 24 GB) ---
MODELO_BASE = "google/mt5-base"   # mejor calidad. Alt: "google/mt5-small" (rápido) o "google/mt5-large" (máxima)
MAX_ENTRADA = 16        # tokens del prompt de temática
MAX_SALIDA  = 400       # largo máximo del cuento en tokens (truncamos los muy largos)
LOTE        = 16        # la 4090 aguanta esto con mt5-base en bf16. Súbelo si ves VRAM libre.
ACUMULAR    = 1         # gradient accumulation: lote efectivo = LOTE * ACUMULAR
EPOCAS      = 3         # como partimos de un modelo preentrenado, bastan pocas
LR          = 3e-4
USAR_BF16   = True      # bf16 mixto (la 4090 lo soporta; es estable para T5). Auto-detecta; si no hay soporte usa fp32.
GRADIENT_CHECKPOINTING = False  # ponlo True solo si usas mt5-large
LIMITE_EJEMPLOS = None  # pon p.ej. 2000 para una prueba rápida; None = todos

print("Repo:", RUTA_REPO)
print("Existe carpeta de parciales:", os.path.isdir(RUTA_PARCIALES))

## Paso 3 — Armar el dataset maestro

Une todos los CSV parciales y quita duplicados exactos (por `hash_texto`). Genera `datos/cuentos/dataset_cuentos.csv`.
Si ese archivo ya existe, esta celda no lo vuelve a crear (borra el archivo si quieres regenerarlo).

In [ ]:
import csv, glob
from collections import defaultdict

csv.field_size_limit(10 * 1024 * 1024)

if os.path.exists(RUTA_DATASET):
    print("Ya existe el dataset maestro:", RUTA_DATASET)
else:
    archivos = sorted(glob.glob(os.path.join(RUTA_PARCIALES, "*.csv")))
    filas = []
    for ruta in archivos:
        with open(ruta, encoding="utf-8") as f:
            filas.extend(csv.DictReader(f))
    print(f"Leidas {len(filas)} filas de {len(archivos)} archivos")

    por_hash = defaultdict(list)
    for fila in filas:
        por_hash[fila["hash_texto"]].append(fila)
    unicos = [grupo[0] for grupo in por_hash.values()]

    os.makedirs(os.path.dirname(RUTA_DATASET), exist_ok=True)
    campos = list(unicos[0].keys())
    with open(RUTA_DATASET, "w", encoding="utf-8", newline="") as f:
        escritor = csv.DictWriter(f, fieldnames=campos)
        escritor.writeheader()
        escritor.writerows(unicos)
    print(f"Dataset maestro: {len(unicos)} cuentos unicos (de {len(filas)}) -> {RUTA_DATASET}")

## Paso 4 — Cargar el dataset en pares (entrada, cuento)

In [ ]:
from torch.utils.data import DataLoader, Dataset

def construir_entrada(tematica):
    """Prompt del encoder. DEBE ser identico al entrenar y al generar."""
    return f"escribe un cuento sobre {tematica.replace('_', ' ')}"

def cargar_dataset(ruta, limite=None):
    pares, tematicas = [], set()
    with open(ruta, encoding="utf-8") as f:
        for fila in csv.DictReader(f):
            texto = fila["texto"].strip()
            tematica = fila["tematica"].strip()
            if texto and tematica:
                pares.append((construir_entrada(tematica), texto))
                tematicas.add(tematica)
            if limite and len(pares) >= limite:
                break
    return pares, sorted(tematicas)

class CuentosDataset(Dataset):
    def __init__(self, pares):
        self.pares = pares
    def __len__(self):
        return len(self.pares)
    def __getitem__(self, i):
        return self.pares[i]

pares, tematicas = cargar_dataset(RUTA_DATASET, LIMITE_EJEMPLOS)
print(f"Cuentos cargados: {len(pares)}  Tematicas: {len(tematicas)}")
print(tematicas)

## Paso 5 — Cargar tokenizer y modelo

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", dispositivo, "| Modelo base:", MODELO_BASE)

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
modelo = AutoModelForSeq2SeqLM.from_pretrained(MODELO_BASE).to(dispositivo)

if GRADIENT_CHECKPOINTING:
    modelo.gradient_checkpointing_enable()
    modelo.config.use_cache = False  # incompatible con checkpointing

def hacer_colar(tokenizer, max_entrada, max_salida):
    def colar(lote):
        entradas = [e for e, _ in lote]
        objetivos = [o for _, o in lote]
        enc = tokenizer(entradas, max_length=max_entrada, truncation=True,
                        padding=True, return_tensors="pt")
        obj = tokenizer(objetivos, max_length=max_salida, truncation=True,
                        padding=True, return_tensors="pt")
        etiquetas = obj["input_ids"]
        etiquetas[etiquetas == tokenizer.pad_token_id] = -100  # el padding no cuenta en la perdida
        enc["labels"] = etiquetas
        return enc
    return colar

cargador = DataLoader(
    CuentosDataset(pares), batch_size=LOTE, shuffle=True,
    collate_fn=hacer_colar(tokenizer, MAX_ENTRADA, MAX_SALIDA),
    num_workers=0, pin_memory=True,  # 0 = sin multiprocessing (Python 3.14 usa forkserver y no puede picklear el colar)
)
print("Lotes por epoca:", len(cargador))

## Paso 6 — Entrenar

Con ~23k cuentos y `google/mt5-base` en bf16 en una 4090, cada época tarda del orden de pocos minutos a ~15 min. Usa `LIMITE_EJEMPLOS` para una prueba rápida primero.

In [ ]:
from tqdm.auto import tqdm

usar_bf16 = USAR_BF16 and torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print("Precision de entrenamiento:", "bf16 (mixto)" if usar_bf16 else "fp32")

optimizador = torch.optim.AdamW(modelo.parameters(), lr=LR)

for epoca in range(1, EPOCAS + 1):
    modelo.train()
    perdida_total = 0.0
    optimizador.zero_grad()
    barra = tqdm(cargador, desc=f"Epoca {epoca}/{EPOCAS}")
    for paso, lote in enumerate(barra, 1):
        lote = {k: v.to(dispositivo) for k, v in lote.items()}
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=usar_bf16):
            salida = modelo(**lote)
            perdida = salida.loss / ACUMULAR
        perdida.backward()
        if paso % ACUMULAR == 0:
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            optimizador.step()
            optimizador.zero_grad()
        perdida_total += salida.loss.item()
        barra.set_postfix(perdida=f"{salida.loss.item():.3f}")
    print(f"Epoca {epoca}/{EPOCAS}  perdida_media={perdida_total / len(cargador):.4f}")
    if torch.cuda.is_available():
        print(f"  VRAM pico: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

## Paso 7 — Guardar el modelo afinado

In [ ]:
import json

os.makedirs(RUTA_SALIDA, exist_ok=True)
if GRADIENT_CHECKPOINTING:
    modelo.config.use_cache = True  # reactivar para generar
modelo.save_pretrained(RUTA_SALIDA)
tokenizer.save_pretrained(RUTA_SALIDA)
with open(os.path.join(RUTA_SALIDA, "tematicas.json"), "w", encoding="utf-8") as f:
    json.dump({"tematicas": tematicas}, f, ensure_ascii=False)
print("Modelo afinado guardado en", RUTA_SALIDA)

## Paso 8 — Generar un cuento de prueba

In [ ]:
def generar(tematica, temperatura=0.9, max_tokens=400):
    modelo.eval()
    entrada = tokenizer(construir_entrada(tematica), return_tensors="pt").to(dispositivo)
    with torch.no_grad():
        salida = modelo.generate(
            **entrada, do_sample=True, temperature=temperatura, top_p=0.95,
            max_length=max_tokens, repetition_penalty=1.3, no_repeat_ngram_size=3,
        )
    return tokenizer.decode(salida[0], skip_special_tokens=True)

print(generar("espacio"))